# Persian Book Translator (Colab)
Runs in Google's cloud, nothing on your PC. Run the cells top to bottom.

In [ ]:
#@title 1. Upload PersianBookTranslator.zip
from google.colab import files
import zipfile, os
up = files.upload()
zipfile.ZipFile(next(iter(up))).extractall('/content')
os.chdir('/content/PersianBookTranslator')
!pip -q install -r requirements.txt

In [ ]:
#@title 2. Your providers (add as many as you have)
# Backup keys: "api_key": "key1, key2, key3" (used one after another when a key runs out)
# "use_for": ["translate"], ["review"] or ["translate", "review"] (missing = both).
#   MODE="translate" only uses providers with "translate"; MODE="refine" (review) only those with "review".
PROVIDERS = [
    {"name": "provider-1", "base_url": "https://YOUR-PROVIDER-1/v1", "api_key": "YOUR_API_KEY_HERE", "model": "MODEL_ID", "concurrency": 1, "use_for": ["translate"]},
    {"name": "provider-2", "base_url": "https://YOUR-PROVIDER-2/v1", "api_key": "YOUR_API_KEY_HERE", "model": "MODEL_ID", "concurrency": 1, "use_for": ["translate", "review"]},
]
INSTRUCTIONS = ""  #@param {type:"string"}
CHUNK_CHARS = 2500  #@param {type:"integer"}
PERSIAN_DIGITS = False  #@param {type:"boolean"}
MODE = "translate"  #@param ["translate", "refine"]
# refine = the book you upload is ALREADY Persian; it gets polished instead of translated

In [ ]:
#@title 3. Test every provider
from ptranslator.providers import ProviderConfig, chat_completion
from ptranslator.prompts import build_system_prompt
for p in PROVIDERS:
    if not ProviderConfig.from_dict(p).used_for(MODE):
        print('SKIP', p['name'], '-> not set for', 'review' if MODE == 'refine' else 'translation'); continue
    try:
        print('OK ', p['name'], '->', chat_completion(ProviderConfig.from_dict(p), build_system_prompt(mode=MODE), 'پیرمرد به دریا نگاه کرد.' if MODE == 'refine' else 'The old man looked at the sea.', timeout=60))
    except Exception as e:
        print('FAIL', p['name'], '->', e)

In [ ]:
#@title 4. Upload your book and translate
import os
os.environ['PBT_DATA_DIR'] = '/content/data'
from ptranslator.job import TranslationJob
from ptranslator.dispatcher import DispatchSettings
book = files.upload()
name = next(iter(book))
job = TranslationJob(name, book[name], [ProviderConfig.from_dict(p) for p in PROVIDERS],
                     DispatchSettings(max_chars=CHUNK_CHARS, extra_instructions=INSTRUCTIONS, persian_digits=PERSIAN_DIGITS, mode=MODE),
                     data_dir='/content/data', log=print)
import threading
runner = threading.Thread(target=job.run, daemon=True); runner.start()
print('Started in the background. Run the last cell to wait for it and download.')
print('If every key runs out, it PAUSES: use cell 5 to add a key/provider and it continues.')


In [ ]:
#@title 5. (Optional, while cell 4 is running) add a new provider or extra key
# Same name as an existing provider = its new keys are added. New name = new provider.
job.add_providers([ProviderConfig.from_dict(
    {"name": "provider-3", "base_url": "https://YOUR-PROVIDER-3/v1", "api_key": "NEW_KEY", "model": "MODEL_ID", "concurrency": 1}
)])


In [ ]:
#@title 6. Watch progress (time left + last translated part) and download
import time
from IPython.display import clear_output
while runner.is_alive():
    p = job.progress(); clear_output(wait=True)
    eta = p['eta_seconds']
    print(f"{p['status']}  {p['percent']}%  time left: {'~%dm %ds' % divmod(eta, 60) if eta is not None else ('paused' if p['status']=='waiting' else 'calculating...')}")
    if p['waiting']: print('PAUSED:', p['wait_reason'], '-> add a key in cell 5')
    if p['last']: print('\n--- last translated part ---\n' + p['last']['translation'][:1500])
    time.sleep(5)
if job.output:
    open(job.output_name, 'wb').write(job.output)
    files.download(job.output_name)
